# Testes de Scraping em Fontes Web

Avaliação da viabilidade de extração de dados das fontes indicadas para possível utilização na plataforma de relatórios agênticos do SenacLabs.

# Objetivo

Avaliar a viabilidade de scraping das fontes web indicadas, verificando:

- disponibilidade e resposta HTTP;
- estrutura do conteúdo retornado;
- possibilidade de extração de dados utilizando Python;
- possíveis mecanismos de bloqueio ou anti-bot;
- possível dependência de JavaScript para carregamento do conteúdo;
- comportamento e estabilidade básica das fontes durante os testes.

O Fluxo de dados extraidos será voltado para a plataforma de criação de relatórios agênticos do SenacLabs


## Dependências

Para os testes serão utilizadas bibliotecas populares do ecossistema Python:

- `requests`: realização de requisições HTTP;
- `BeautifulSoup`: leitura e análise do HTML;
- `dataclasses`: estruturação dos resultados;
- `urllib.parse`: manipulação das URLs;
- `time`: medição do tempo de resposta.

In [2]:
from dataclasses import dataclass
from urllib.parse import urlparse
import time

import requests
from bs4 import BeautifulSoup

## Estruturas auxiliares

Os resultados de cada teste serão armazenados em uma estrutura padronizada para facilitar a comparação entre as fontes.

In [3]:
@dataclass
class TestResult:
    url: str
    status_code: int | None
    response_time: float | None
    content_length: int | None
    content_type: str | None
    title: str | None
    robots_status: int | None
    possible_block: bool
    block_indicators: list[str]
    possible_js_required: bool
    error: str | None

A classe abaixo concentra os testes básicos realizados em cada fonte.

In [4]:
class SourceTester:
    def __init__(self):
        self.session = requests.Session()

        self.session.headers.update({
            "User-Agent": "SourceCompatibilityTester/1.0"
        })

    def test(self, url: str) -> TestResult:
        print(f"\n{'=' * 60}")
        print(f"Testando: {url}")
        print(f"{'=' * 60}")

        robots_status = self._check_robots(url)

        try:
            start = time.perf_counter()

            response = self.session.get(
                url,
                timeout=15,
                allow_redirects=True
            )

            elapsed = time.perf_counter() - start

            content_type = response.headers.get("Content-Type")

            print(f"Status: {response.status_code}")
            print(f"URL final: {response.url}")
            print(f"Tempo: {elapsed:.2f}s")
            print(f"Tamanho: {len(response.content)} bytes")
            print(f"Content-Type: {content_type}")

            title = self._extract_title(response.text)

            block_indicators = self._detect_block_reasons(response)

            blocked = (
                response.status_code in {401, 403, 429, 503}
                or len(block_indicators) > 0
            )

            js_required = self._detect_possible_js_page(response.text)

            print(f"Título: {title}")
            print(f"Possível bloqueio: {blocked}")
            print(f"Possível dependência de JS: {js_required}")

            return TestResult(
                url=url,
                status_code=response.status_code,
                response_time=elapsed,
                content_length=len(response.content),
                content_type=content_type,
                title=title,
                robots_status=robots_status,
                possible_block=blocked,
                block_indicators=block_indicators,
                possible_js_required=js_required,
                error=None
            )

        except requests.RequestException as exc:
            print(f"Erro de requisição: {exc}")

            return TestResult(
                url=url,
                status_code=None,
                response_time=None,
                content_length=None,
                content_type=None,
                title=None,
                robots_status=robots_status,
                possible_block=False,
                block_indicators=[],
                possible_js_required=False,
                error=str(exc)
            )

    def _check_robots(self, url: str) -> int | None:
        parsed = urlparse(url)

        robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"

        try:
            response = self.session.get(
                robots_url,
                timeout=10
            )

            print(f"robots.txt: {response.status_code}")

            return response.status_code

        except requests.RequestException:
            print("robots.txt: não foi possível consultar.")
            return None

    @staticmethod
    def _extract_title(html: str) -> str | None:
        soup = BeautifulSoup(html, "html.parser")

        if soup.title and soup.title.string:
            return soup.title.string.strip()

        return None

    @staticmethod
    def _detect_block_reasons(response: requests.Response) -> list[str]:
        text = response.text.lower()

        indicators = [
            "access denied",
            "too many requests",
            "captcha",
            "verify you are human",
            "checking your browser",
        ]

        return [
            indicator
            for indicator in indicators
            if indicator in text
        ]

    @staticmethod
    def _detect_possible_js_page(html: str) -> bool:
        soup = BeautifulSoup(html, "html.parser")

        text = soup.get_text(" ", strip=True)

        scripts = soup.find_all("script")

        return len(text) < 200 and len(scripts) >= 5


## Fontes avaliadas

As fontes abaixo foram indicadas para os testes. As três primeiras possuem prioridade.

In [5]:
sources = [
    "https://www.camara.leg.br/busca-portal/proposicoes/pesquisa-simplificada",
    "https://www.gov.br/anvisa/pt-br",
    "https://www.gov.br/imprensanacional/pt-br",
    "https://www25.senado.leg.br/web/atividade",
    "https://www.al.rn.leg.br",
    "https://deirn.sdoe.com.br",
    "https://www.cmnat.rn.gov.br",
    "https://legislativo.camaraparnamirim.rn.gov.br/",
    "https://www.mossoro.rn.leg.br",
    "https://www.gov.br/anvisa",
    "https://www.gov.br/conitec",
    "https://www.gov.br/saude",
    "https://www.confaz.fazenda.gov.br",
    "https://www.gov.br/cade",
    "https://www.gov.br/senacon"
]

## Teste inicial de acessibilidade

Nesta etapa são verificadas características básicas das fontes, como:

- código HTTP;
- tempo de resposta;
- tamanho do conteúdo;
- tipo do conteúdo retornado;
- título da página;
- disponibilidade do `robots.txt`;
- possíveis indícios de bloqueio;
- possível dependência de JavaScript.

In [6]:
tester = SourceTester()

results = []

for source in sources:
    result = tester.test(source)
    results.append(result)

    time.sleep(2)


Testando: https://www.camara.leg.br/busca-portal/proposicoes/pesquisa-simplificada
robots.txt: 200
Status: 200
URL final: https://www.camara.leg.br/busca-portal/proposicoes/pesquisa-simplificada
Tempo: 0.11s
Tamanho: 32131 bytes
Content-Type: text/html; charset=utf-8
Título: Portal da Câmara dos Deputados - Busca - Portal da Câmara dos Deputados
Possível bloqueio: False
Possível dependência de JS: False

Testando: https://www.gov.br/anvisa/pt-br
robots.txt: 200
Status: 200
URL final: https://www.gov.br/anvisa/pt-br
Tempo: 0.57s
Tamanho: 579734 bytes
Content-Type: text/html; charset=utf-8
Título: Português (Brasil)
Possível bloqueio: True
Possível dependência de JS: False

Testando: https://www.gov.br/imprensanacional/pt-br
robots.txt: 200
Status: 200
URL final: https://www.gov.br/imprensanacional/pt-br
Tempo: 0.20s
Tamanho: 324547 bytes
Content-Type: text/html; charset=utf-8
Título: Imprensa Nacional
Possível bloqueio: True
Possível dependência de JS: False

Testando: https://www25.se

## Resultados consolidados

Os resultados obtidos nos testes iniciais são apresentados abaixo para facilitar a comparação entre as fontes

In [7]:
import pandas as pd

df_results = pd.DataFrame(
    [vars(result) for result in results]
)

df_results

,url,status_code,response_time,content_length,content_type,title,robots_status,possible_block,block_indicators,possible_js_required,error
0,https://www.camara.leg.br/busca-portal/proposi...,200,0.112330,32131,text/html; charset=utf-8,Portal da Câmara dos Deputados - Busca - Porta...,200,False,[],False,None
1,https://www.gov.br/anvisa/pt-br,200,0.571249,579734,text/html; charset=utf-8,Português (Brasil),200,True,[captcha],False,None
2,https://www.gov.br/imprensanacional/pt-br,200,0.196495,324547,text/html; charset=utf-8,Imprensa Nacional,200,True,[captcha],False,None
3,https://www25.senado.leg.br/web/atividade,200,0.340000,83672,text/html;charset=UTF-8,Home - Senado Federal,200,False,[],False,None
4,https://www.al.rn.leg.br,200,0.514963,87272,text/html; charset=UTF-8,Assembleia Legislativa do Rio Grande do Norte,200,True,[captcha],False,None
5,https://deirn.sdoe.com.br,403,0.132656,1233,text/html,403 - Forbidden: Access is denied.,404,True,[],False,None
6,https://www.cmnat.rn.gov.br,200,0.147953,73347,text/html; charset=UTF-8,Câmara Municipal de Natal,404,True,[captcha],False,None
7,https://legislativo.camaraparnamirim.rn.gov.br/,200,0.840448,198808,text/html; charset=utf-8,Câmara Municipal de Parnamirim - Portal Oficia...,200,True,[captcha],False,None
8,https://www.mossoro.rn.leg.br,200,0.105769,57351,text/html;charset=utf-8,CMM — Câmara Municipal de Mossoró,200,False,[],False,None
9,https://www.gov.br/anvisa,200,0.536139,579734,text/html; charset=utf-8,Português (Brasil),200,True,[captcha],False,None


## Investigação de possíveis bloqueios

In [8]:
def detect_block_reason(response):
    text = response.text.lower()

    indicators = [
        "access denied",
        "too many requests",
        "captcha",
        "verify you are human",
        "checking your browser",
    ]

    return [
        indicator
        for indicator in indicators
        if indicator in text
    ]

In [9]:
suspected_sources = [
    "https://www.gov.br/anvisa/pt-br",
    "https://www.gov.br/imprensanacional/pt-br",
]

for source in suspected_sources:
    response = tester.session.get(source, timeout=15)

    print(source)
    print(detect_block_reason(response))
    print()

https://www.gov.br/anvisa/pt-br
['captcha']

https://www.gov.br/imprensanacional/pt-br
['captcha']



## Teste de extração

### Câmara dos Deputados

Será realizada uma inspeção inicial do HTML retornado para verificar se existem elementos e conteúdos que possam ser extraídos diretamente.

In [10]:
url = sources[0]

response = requests.get(url, timeout=15)

soup = BeautifulSoup(response.text, "html.parser")

print("Título:", soup.title.string if soup.title else None)
print("Links encontrados:", len(soup.find_all("a")))

Título: 
      Portal da Câmara dos Deputados - Busca - Portal da Câmara dos Deputados
  
Links encontrados: 70


In [11]:
for link in soup.find_all("a")[:10]:
    text = link.get_text(" ", strip=True)

    if text:
        print(text, "->", link.get("href"))

Ir ao conteúdo -> #main-content
Ir à navegação principal -> #main-nav
Página inicial -> https://www.camara.leg.br
Acessibilidade -> https://www2.camara.leg.br/acessibilidade/recursos-de-acessibilidade
Fale Conosco -> https://www.camara.leg.br/fale-conosco
Congresso -> http://www.congressonacional.leg.br
Senado -> http://www.senado.leg.br
English EN -> https://www2.camara.leg.br/english
Español ES -> https://www2.camara.leg.br/espanol
Entrar -> https://www.camara.leg.br/login


## Recomendações

Com base nos testes realizados:

- Fontes que retornam os dados diretamente no HTML podem utilizar `requests` e `BeautifulSoup`;
- Fontes cujo conteúdo é carregado dinamicamente devem ter suas requisições de rede analisadas para identificar possíveis APIs ou endpoints;
- Selenium ou Playwright devem ser considerados apenas quando a extração depender efetivamente da execução de JavaScript;
- Os seletores utilizados devem ser mantidos isolados para facilitar futuras atualizações caso a estrutura HTML das fontes seja alterada.

## Conclusão

Os testes permitiram identificar o comportamento inicial das fontes analisadas e avaliar a viabilidade de utilização de técnicas de scraping.

Os resultados serão postos aqui após a confirmação de todas as análises completadas.